# rank-world-size-args — faded example 3: Source sends world_size-1 messages, not world_size

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `rank-world-size-args`. The last cell reports your progress on the `Distributed: rank/world_size args` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: rank/world_size args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rank-world-size-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rank-world-size-args"
DD_SUBTOPIC = "Distributed: rank/world_size args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When the source rank broadcasts, it sends to exactly `world_size - 1` other ranks — it does not send to itself. This means the length of the source's action list is `world_size - 1`, not `world_size`. A common off-by-one error is iterating `range(world_size)` without excluding `src`, which would generate a spurious `('send', src)` self-message.

## Faded exercise 3

### Exercise — Count the correct fanout from source

Complete `count_broadcast_fanout(world_size, src=0)`. Call `broadcast_protocol(None, src, world_size, src=src)` and return the number of send actions produced.

Fill in the fanout count expression.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

def broadcast_protocol(tensor, rank, world_size, src=0):
    if rank == src:
        return [('send', other) for other in range(world_size) if other != src]
    return [('recv', src)]

def count_broadcast_fanout(world_size: int, src: int = 0) -> int:
    actions = broadcast_protocol(None, src, world_size, src=src)
    return sum(1 for a in actions if a[0] == 'send')

for ws in [2, 4, 8]:
    print(f'world_size={ws}: fanout={count_broadcast_fanout(ws)}')


def _test():
    for ws in [1, 2, 4, 8]:
        expected = ws - 1
        got = count_broadcast_fanout(ws)
        assert got == expected, f'world_size={ws}: expected {expected} sends, got {got}'
    # Non-zero src
    for ws in [3, 5]:
        got = count_broadcast_fanout(ws, src=1)
        assert got == ws - 1


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def broadcast_protocol(tensor, rank, world_size, src=0):
    if rank == src:
        return [('send', other) for other in range(world_size) if other != src]
    return [('recv', src)]

def count_broadcast_fanout(world_size: int, src: int = 0) -> int:
    actions = broadcast_protocol(None, src, world_size, src=src)
    return sum(1 for a in actions if a[0] == 'send')

for ws in [2, 4, 8]:
    print(f'world_size={ws}: fanout={count_broadcast_fanout(ws)}')
```
</details>